# 04. Train The Student With MLX-LM

This notebook trains the small MLX student on the teacher trajectories from notebook 03.

The training objective is offline token-level imitation: given the conversation history and tool schemas, train the student to produce the teacher's next assistant message. We mask the prompt, so loss is only computed on the final teacher response.


In [ ]:
from __future__ import annotations

from statistics import mean
import json
import os
import subprocess

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import sft_rows

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_USER_SIMULATOR_LLM = cfg.required_env("TAU_BENCH_USER_SIMULATOR_LLM")
USER_SIMULATOR_SLUG = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
teacher_config = cfg.teacher_config_from_env(default_provider="vllm_raw", default_model=cfg.TEACHER_MODEL)
TEACHER_MODEL = teacher_config.model_name
TEACHER_SLUG = cfg.filename_slug(TEACHER_MODEL)
TEACHER_PROVIDER = teacher_config.provider
STUDENT_MODEL = cfg.STUDENT_MODEL
STUDENT_SLUG = cfg.filename_slug(STUDENT_MODEL)

SFT_WORK_DIR = OUTPUT_DIR / f"{STUDENT_SLUG}_tau3_retail_sft_mlx_lm"
MLX_DATA_DIR = SFT_WORK_DIR / "mlx_lm_data"
ADAPTER_OUTPUT_DIR = SFT_WORK_DIR / f"{STUDENT_SLUG}_tau3_retail_mlx_lora_adapter"
SFT_SOURCE_PATH = OUTPUT_DIR / f"{TEACHER_SLUG}_{TEACHER_PROVIDER}_tau3_bench_retail_train_successful_sft_chat_rows_{USER_SIMULATOR_SLUG}.jsonl"

print("Project root:", ROOT)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("SFT source path:", SFT_SOURCE_PATH)
print("Work dir:", SFT_WORK_DIR)


## Load Teacher Rows

Run notebook 03 first. It creates `messages + tools` JSONL rows from successful teacher trajectories.


In [ ]:
if not SFT_SOURCE_PATH.exists():
    candidates = sorted(OUTPUT_DIR.glob("*_tau3_bench_retail_train_successful_sft_chat_rows_*.jsonl"))
    print("Could not find the expected SFT source path.")
    print("Expected:", SFT_SOURCE_PATH)
    print("Candidate SFT files:")
    for candidate in candidates:
        print(" -", candidate)
    raise FileNotFoundError(SFT_SOURCE_PATH)

rows = cfg.load_jsonl(SFT_SOURCE_PATH)
if not rows:
    raise RuntimeError("The SFT source file exists but contains no rows.")

print("Loaded SFT rows:", len(rows))
print("Unique task ids:", len({row['task_id'] for row in rows}))
print("Tool-call target rows:", sum(row.get("is_tool_call_target", False) for row in rows))
print("Natural-language target rows:", sum(not row.get("is_tool_call_target", False) for row in rows))
print("First row roles:", [message["role"] for message in rows[0]["messages"][-5:]])
print("\nFirst target preview:\n", rows[0]["target_text"][:800])


## Measure Sequence Lengths

MLX-LM will apply Qwen's chat template and include the retail tools in the rendered prompt. We measure that exact training sequence here.

The first verified Mac-native smoke config is conservative:

- student base: `mlx-community/Qwen3.5-0.8B-MLX-bf16` (bf16, non-quantized)
- LoRA layers: 1
- batch size: 1
- prompt masking: on
- tested length: about 4k tokens in the smoke run
- memory observed in the smoke run: about 8.6 GB peak

Longer rows are real data, but they need a stronger verified context/memory setting. We do not silently truncate them for the first run. If memory becomes the blocker, the 4-bit MLX checkpoint is a fallback, not the default path.


In [ ]:
tokenizer = cfg.load_tokenizer()
length_rows = []
for index, row in enumerate(rows):
    token_count = sft_rows.mlx_chat_row_token_length(row, tokenizer)
    length_rows.append({
        "index": index,
        "task_id": row["task_id"],
        "source_message_index": row["source_message_index"],
        "tokens": token_count,
        "is_tool_call_target": row.get("is_tool_call_target", False),
    })

lengths = [item["tokens"] for item in length_rows]
length_stats = {
    "min": min(lengths),
    "p50": cfg.percentile_int(lengths, 0.50),
    "p90": cfg.percentile_int(lengths, 0.90),
    "p95": cfg.percentile_int(lengths, 0.95),
    "max": max(lengths),
    "mean": round(mean(lengths), 1),
}
print(json.dumps(length_stats, indent=2))
print()
print("Longest rows:")
for item in sorted(length_rows, key=lambda item: item["tokens"], reverse=True)[:10]:
    print(item)


## Choose The First MLX Training Config

This cell makes the memory tradeoff explicit. If you want to attempt longer-context training, increase `TAU_SFT_MAX_SEQ_LENGTH` and rerun. The smoke-tested default keeps rows at or below 4,200 tokens and drops longer rows rather than truncating them.

Dropping long rows is not a claim that long rows are unimportant. It is the first stable local training pass. The right next improvement is a larger verified context config or a data strategy that shortens the tool/policy prefix without changing semantics.


In [ ]:
MAX_SEQ_LENGTH = int(os.getenv("TAU_SFT_MAX_SEQ_LENGTH", "4200"))
LORA_NUM_LAYERS = int(os.getenv("TAU_SFT_LORA_NUM_LAYERS", "1"))
LORA_RANK = int(os.getenv("TAU_SFT_LORA_RANK", "8"))
LORA_SCALE = float(os.getenv("TAU_SFT_LORA_SCALE", "20.0"))
LEARNING_RATE = float(os.getenv("TAU_SFT_LEARNING_RATE", "1e-5"))
BATCH_SIZE = int(os.getenv("TAU_SFT_BATCH_SIZE", "1"))
ITERS = int(os.getenv("TAU_SFT_ITERS", "-1"))
GRAD_ACCUMULATION_STEPS = int(os.getenv("TAU_SFT_GRAD_ACCUMULATION_STEPS", "1"))
VALIDATION_TASK_FRACTION = float(os.getenv("TAU_SFT_VALIDATION_TASK_FRACTION", "0.10"))
SPLIT_SEED = int(os.getenv("TAU_SFT_SPLIT_SEED", "42"))

rows_that_fit = [row for row, meta in zip(rows, length_rows) if meta["tokens"] <= MAX_SEQ_LENGTH]
rows_too_long = [meta for meta in length_rows if meta["tokens"] > MAX_SEQ_LENGTH]

print("Student MLX model:", STUDENT_MODEL)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("LoRA layers:", LORA_NUM_LAYERS)
print("LoRA rank:", LORA_RANK)
print("LoRA scale:", LORA_SCALE)
print("Rows kept:", len(rows_that_fit))
print("Rows longer than max sequence length:", len(rows_too_long))

if rows_too_long:
    print("\nLongest dropped rows:")
    for item in sorted(rows_too_long, key=lambda item: item["tokens"], reverse=True)[:10]:
        print(item)

if not rows_that_fit:
    raise RuntimeError("No rows fit the current MAX_SEQ_LENGTH. Increase TAU_SFT_MAX_SEQ_LENGTH or collect shorter rows.")


## Split By Task And Write MLX-LM Data Files

We split by task id, not by row. That prevents later turns from the same solved trajectory leaking into validation.


In [ ]:
train_rows, validation_rows, validation_task_ids = sft_rows.split_sft_rows_by_task_id(
    rows_that_fit,
    validation_task_fraction=VALIDATION_TASK_FRACTION,
    seed=SPLIT_SEED,
)
if not validation_rows:
    validation_rows = train_rows[:1]

MLX_DATA_DIR.mkdir(parents=True, exist_ok=True)
cfg.write_jsonl(MLX_DATA_DIR / "train.jsonl", train_rows)
cfg.write_jsonl(MLX_DATA_DIR / "valid.jsonl", validation_rows)
cfg.write_jsonl(MLX_DATA_DIR / "test.jsonl", validation_rows)

print("Train rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
print("Validation task ids:", sorted(validation_task_ids))
print("MLX data dir:", MLX_DATA_DIR)


## Train With MLX-LM

`--mask-prompt` is the important flag. It tells MLX-LM to compute loss only on the final assistant response in each row.

`HF_HUB_DISABLE_XET=1` avoids the stalled Hugging Face Xet download behavior we saw during the smoke test.


In [ ]:
if ITERS <= 0:
    ITERS = max(1, len(train_rows) // max(1, BATCH_SIZE))

LORA_CONFIG_PATH = SFT_WORK_DIR / "mlx_lm_lora_config.yaml"
SFT_WORK_DIR.mkdir(parents=True, exist_ok=True)
LORA_CONFIG_PATH.write_text(
    "lora_parameters:\n"
    f"  rank: {LORA_RANK}\n"
    f"  scale: {LORA_SCALE}\n"
    "  dropout: 0.0\n",
    encoding="utf-8",
)

train_command = [
    "uv", "run", "python", "-m", "mlx_lm", "lora",
    "--model", STUDENT_MODEL,
    "--train",
    "--data", str(MLX_DATA_DIR),
    "--adapter-path", str(ADAPTER_OUTPUT_DIR),
    "-c", str(LORA_CONFIG_PATH),
    "--iters", str(ITERS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accumulation-steps", str(GRAD_ACCUMULATION_STEPS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--mask-prompt",
    "--num-layers", str(LORA_NUM_LAYERS),
    "--learning-rate", str(LEARNING_RATE),
    "--steps-per-report", "1",
    "--steps-per-eval", str(max(1, min(25, ITERS))),
    "--save-every", str(max(1, min(25, ITERS))),
    "--val-batches", "1",
    "--grad-checkpoint",
    "--clear-cache-threshold", "0.1",
]

print("Training command:")
print("HF_HUB_DISABLE_XET=1 " + " ".join(train_command))

run_training = os.getenv("TAU_SFT_SKIP_TRAINING", "0") != "1"
if run_training:
    env = {**os.environ, "HF_HUB_DISABLE_XET": "1"}
    subprocess.run(train_command, cwd=str(ROOT), env=env, check=True)
else:
    print("Skipping training because TAU_SFT_SKIP_TRAINING=1")


## Save Training Metadata


In [ ]:
TRAINING_METADATA_PATH = SFT_WORK_DIR / "training_metadata.json"
training_metadata = {
    "student_base_model": STUDENT_MODEL,
    "teacher_model": TEACHER_MODEL,
    "sft_source_path": str(SFT_SOURCE_PATH),
    "mlx_data_dir": str(MLX_DATA_DIR),
    "adapter_output_dir": str(ADAPTER_OUTPUT_DIR),
    "raw_rows": len(rows),
    "rows_kept": len(rows_that_fit),
    "rows_dropped_over_length": len(rows_too_long),
    "length_stats": length_stats,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_num_layers": LORA_NUM_LAYERS,
    "lora_rank": LORA_RANK,
    "lora_scale": LORA_SCALE,
    "lora_config_path": str(LORA_CONFIG_PATH),
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "iters": ITERS,
    "adapter_exists": (ADAPTER_OUTPUT_DIR / "adapters.safetensors").exists(),
    "adapter_config_exists": (ADAPTER_OUTPUT_DIR / "adapter_config.json").exists(),
}
SFT_WORK_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_METADATA_PATH.write_text(json.dumps(cfg.make_json_safe(training_metadata), indent=2), encoding="utf-8")
print(json.dumps(training_metadata, indent=2))
print("Saved metadata to:", TRAINING_METADATA_PATH)


## Smoke Test Adapter Loading

This does not replace the held-out benchmark eval. It only checks that MLX-LM can load the trained adapter and generate from one training-shaped prompt.


In [ ]:
from mlx_lm import generate as mlx_generate
from mlx_lm import load as mlx_load
from mlx_lm.sample_utils import make_sampler

if not (ADAPTER_OUTPUT_DIR / "adapters.safetensors").exists():
    raise FileNotFoundError(ADAPTER_OUTPUT_DIR / "adapters.safetensors")

smoke_row = validation_rows[0]
model, mlx_tokenizer = mlx_load(STUDENT_MODEL, adapter_path=str(ADAPTER_OUTPUT_DIR))
smoke_prompt = mlx_tokenizer.apply_chat_template(
    smoke_row["messages"][:-1],
    tools=smoke_row.get("tools"),
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
smoke_text = mlx_generate(
    model,
    mlx_tokenizer,
    prompt=smoke_prompt,
    max_tokens=128,
    sampler=make_sampler(temp=0.0, top_p=1.0, top_k=0),
    verbose=False,
)
print("Expected teacher target:\n")
print(smoke_row["target_text"][:1000])
print("\nStudent with MLX adapter generated:\n")
print(smoke_text)


## What This Notebook Produces

- MLX-LM train data: printed as `MLX_DATA_DIR` above.
- MLX LoRA adapter: printed as `ADAPTER_OUTPUT_DIR` above.
- Metadata: printed as `TRAINING_METADATA_PATH` above.

Notebook 05 loads that adapter and runs the held-out τ³-Bench retail eval again.
